# Qwen2.5 1.5B QLoRA for ECtHR Semantic-Tree Traversal

This notebook is configured as a tiny smoke test for the actual LATTICE-style tree decision task.

Instead of training only `facts -> alleged articles`, it creates many supervised examples per ECtHR case:

`case facts + current tree path + child semantic summaries -> rank child nodes`

The gold child choices are derived from `AUEB-NLP/ecthr_cases`, config `alleged-violation-prediction`, by mapping each case's alleged article labels to article leaves in the EU Convention semantic tree.

## 1. Imports and Paths

In [ ]:
from pathlib import Path
import gc
import json
import random
import re
import sys

import torch
from datasets import Dataset, load_dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    DataCollatorForSeq2Seq,
    Trainer,
    TrainingArguments,
)

NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / 'local_retrieval_model.py').exists():
    for candidate in [
        NOTEBOOK_DIR / 'llm-guided-retrieval' / 'src' / 'llm_rl_playground',
        NOTEBOOK_DIR / 'src' / 'llm_rl_playground',
        NOTEBOOK_DIR.parent / 'llm_rl_playground',
    ]:
        if (candidate / 'local_retrieval_model.py').exists():
            NOTEBOOK_DIR = candidate
            break

if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

print(f'Using notebook directory: {NOTEBOOK_DIR}')
print(f'CUDA available: {torch.cuda.is_available()}')

## 2. Config

In [ ]:
MODEL_ID = 'Qwen/Qwen2.5-1.5B-Instruct'

ECTHR_DATASET = 'AUEB-NLP/ecthr_cases'
ECTHR_CONFIG = 'alleged-violation-prediction'
ECTHR_TRAIN_SPLIT = 'train'
ECTHR_EVAL_SPLIT = 'validation'

TREE_PATH = NOTEBOOK_DIR.parents[1] / 'trees' / 'EU' / 'eu_conventions_notebook' / 'eu_conventions_tree-bottom-up-llm.json'
OUTPUT_DIR = NOTEBOOK_DIR / 'outputs' / 'qwen2.5-1.5b-ecthr-tree-traversal-qlora'

SEED = 42
MAX_TRAIN_CASES = 8
MAX_EVAL_CASES = 4
MAX_EXAMPLES_PER_CASE = 3
MAX_TRAIN_EXAMPLES = 24
MAX_EVAL_EXAMPLES = 12
MAX_FACT_CHARS = 9000
MAX_CHILD_DESC_CHARS = 1100
MAX_PATH_CHARS = 1800
MAX_LENGTH = 2048

# Single-child nodes are easy and can dominate the dataset, so the default focuses on real branching decisions.
INCLUDE_SINGLE_CHILD_NODES = False

NUM_TRAIN_EPOCHS = 0.25
LEARNING_RATE = 2e-4
PER_DEVICE_TRAIN_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 8

LORA_R = 16
LORA_ALPHA = 16
LORA_DROPOUT = 0.05

# Set this to OUTPUT_DIR / 'checkpoint-*' to resume the same run with optimizer state.
RESUME_FROM_CHECKPOINT = None

assert TREE_PATH.exists(), f'Missing tree file: {TREE_PATH}'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print({'model': MODEL_ID, 'ecthr_config': ECTHR_CONFIG, 'tree': str(TREE_PATH), 'output': str(OUTPUT_DIR)})

## 3. Article and Case Helpers

In [ ]:
def clean_space(value):
    return ' '.join(str(value or '').split())


def normalize_article_label(value):
    if value is None:
        return None
    s = str(value).strip().lower()
    if not s:
        return None

    s = s.replace('no violation', '')
    s = s.replace('non violation', '')
    s = s.replace('non-violation', '')
    s = s.replace('.', '')
    s = s.replace('-', '_')
    s = re.sub(r'\s+', '_', s)
    s = re.sub(r'^echr_', '', s)
    s = re.sub(r'^convention_', '', s)

    if re.fullmatch(r'\d+[a-z]?', s):
        return f'article_{s}'

    m = re.fullmatch(r'p(\d+)_(\d+)', s)
    if m:
        return f'protocol_{int(m.group(1))}_article_{int(m.group(2))}'

    m = re.search(r'article_?(\d+[a-z]?)_(?:of_)?protocol_?(\d+)', s)
    if m:
        return f'protocol_{int(m.group(2))}_article_{m.group(1)}'

    m = re.search(r'protocol_?(\d+)_article_?(\d+[a-z]?)', s)
    if m:
        return f'protocol_{int(m.group(1))}_article_{m.group(2)}'

    m = re.search(r'article_?(\d+[a-z]?)', s)
    if m:
        return f'article_{m.group(1)}'

    return None


def article_id_to_display(article_id):
    m = re.fullmatch(r'article_(\d+[a-z]?)', article_id)
    if m:
        return f'Article {m.group(1).upper()}'
    m = re.fullmatch(r'protocol_(\d+)_article_(\d+[a-z]?)', article_id)
    if m:
        return f'Protocol {m.group(1)}, Article {m.group(2).upper()}'
    return article_id


TREE_ARTICLE_RE = re.compile(
    r'(?:Protocol\s+(?P<protocol>\d+)\s+)?Art\.\s*(?P<article>\d+[A-Za-z]?)',
    flags=re.IGNORECASE,
)


def extract_articles_from_tree_text(text):
    articles = set()
    for match in TREE_ARTICLE_RE.finditer(text or ''):
        article = match.group('article').lower()
        protocol = match.group('protocol')
        if protocol:
            articles.add(f'protocol_{int(protocol)}_article_{article}')
        else:
            articles.add(f'article_{article}')
    return articles


def example_gold_articles(example, label_column='labels'):
    raw_labels = example.get(label_column, [])
    if raw_labels is None:
        return []
    if isinstance(raw_labels, (int, str)):
        raw_labels = [raw_labels]
    normalized = []
    for raw_label in raw_labels:
        article_id = normalize_article_label(raw_label)
        if article_id and article_id not in normalized:
            normalized.append(article_id)
    return sorted(normalized)


def facts_to_text(example, max_chars=MAX_FACT_CHARS):
    facts = example.get('text') or example.get('facts') or []
    if isinstance(facts, str):
        fact_text = facts
    else:
        fact_text = '\n'.join(f'- {clean_space(fact)}' for fact in facts if clean_space(fact))
    if len(fact_text) > max_chars:
        fact_text = fact_text[:max_chars] + '\n- [facts truncated]'
    return fact_text


def node_label(node):
    meta = node.get('metadata') or {}
    payload = meta.get('summary_payload') or {}
    return clean_space(payload.get('label') or meta.get('label') or node.get('id') or 'node')

## 4. Load and Annotate the Semantic Tree

Each node gets a private `_subtree_articles` set containing all normalized article IDs found in descendant leaves. A child is a gold child when its subtree contains one of the case's alleged labels.

In [ ]:
with TREE_PATH.open('r', encoding='utf-8') as f:
    tree = json.load(f)


def annotate_subtree_articles(node):
    children = node.get('child') or []
    if not children:
        articles = extract_articles_from_tree_text(node.get('desc', ''))
    else:
        articles = set()
        for child in children:
            articles |= annotate_subtree_articles(child)
    node['_subtree_articles'] = sorted(articles)
    return articles


all_tree_articles = annotate_subtree_articles(tree)
print(f'Tree article IDs: {len(all_tree_articles)}')
print(sorted(all_tree_articles)[:40])

## 5. Build Traversal Training Examples

In [ ]:
def format_child_options(children):
    parts = []
    for idx, child in enumerate(children):
        desc = clean_space(child.get('desc'))[:MAX_CHILD_DESC_CHARS]
        parts.append(f'[{idx}]. {desc}')
    return '\n\n'.join(parts)


def format_current_path(path_labels):
    text = ' -> '.join(path_labels)
    if len(text) > MAX_PATH_CHARS:
        text = text[-MAX_PATH_CHARS:]
    return text or 'ROOT'


def build_traversal_prompt(facts, path_labels, children):
    valid_ids = ', '.join(str(i) for i in range(len(children)))
    return f"""You are an intelligent search agent navigating a hierarchical semantic tree of European Convention law.

Given the facts of an ECtHR case, choose the child nodes most likely to lead to the Convention or Protocol articles alleged by the applicant.

Relevance definition:
A child node is relevant when its subtree is a strong legal path toward an article that could be alleged from the case facts. Penalize nodes that are only background, procedural context, or weakly related.

Case facts:
{facts}

Current tree path:
{format_current_path(path_labels)}

Candidate child nodes:
Valid candidate IDs for this request: {valid_ids}.

{format_child_options(children)}

Return one clean JSON object with exactly these keys: reasoning, ranking, relevance_scores.
The ranking must include only valid candidate IDs, ordered from most to least relevant.
The relevance_scores field must be an array of [candidate_id, score] pairs with scores from 0 to 100."""


def make_traversal_answer(positive_ids, children, gold_articles):
    ranking = list(positive_ids) + [idx for idx in range(len(children)) if idx not in set(positive_ids)]
    scores = [[idx, 95 if idx in set(positive_ids) else 15] for idx in ranking]
    selected_labels = [node_label(children[idx]) for idx in positive_ids]
    gold_display = [article_id_to_display(article_id) for article_id in sorted(gold_articles)]
    return {
        'reasoning': (
            'The top-ranked child nodes are on oracle paths from the case facts to the alleged article labels. '
            f'Selected child labels: {selected_labels}. Alleged labels: {gold_display}.'
        ),
        'ranking': ranking,
        'relevance_scores': scores,
    }


def collect_case_traversal_examples(node, facts, gold_articles, path_labels, rows):
    children = node.get('child') or []
    if not children:
        return

    positive_ids = [
        idx for idx, child in enumerate(children)
        if set(child.get('_subtree_articles') or []) & gold_articles
    ]
    if not positive_ids:
        return

    if INCLUDE_SINGLE_CHILD_NODES or len(children) > 1:
        prompt = build_traversal_prompt(facts=facts, path_labels=path_labels, children=children)
        answer = make_traversal_answer(positive_ids, children, gold_articles)
        rows.append({
            'messages': [
                {'role': 'system', 'content': 'You are an ECtHR semantic-tree traversal model. Return only valid JSON.'},
                {'role': 'user', 'content': prompt},
                {'role': 'assistant', 'content': json.dumps(answer, ensure_ascii=False)},
            ]
        })

    next_path = path_labels + [node_label(node)]
    for idx in positive_ids:
        collect_case_traversal_examples(children[idx], facts, gold_articles, next_path, rows)


def make_examples_for_case(example, rng):
    gold_articles = set(example_gold_articles(example))
    gold_articles &= set(all_tree_articles)
    if not gold_articles:
        return []

    rows = []
    collect_case_traversal_examples(tree, facts_to_text(example), gold_articles, ['ROOT'], rows)
    rng.shuffle(rows)
    return rows[:MAX_EXAMPLES_PER_CASE]


def build_traversal_rows(dataset, max_cases, max_rows, seed):
    rng = random.Random(seed)
    indexes = list(range(len(dataset)))
    rng.shuffle(indexes)

    rows = []
    cases_used = 0
    skipped_cases = 0
    for idx in indexes[:max_cases]:
        case_rows = make_examples_for_case(dataset[idx], rng)
        if not case_rows:
            skipped_cases += 1
            continue
        rows.extend(case_rows)
        cases_used += 1
        if len(rows) >= max_rows:
            rows = rows[:max_rows]
            break
    return rows, {'cases_used': cases_used, 'skipped_cases': skipped_cases, 'rows': len(rows)}

## 6. Load ECtHR Cases and Generate Rows

The split happens at the case level first, then each case is expanded into multiple node-decision examples. This avoids leaking the same case into both train and eval.

In [ ]:
train_cases = load_dataset(
    ECTHR_DATASET,
    ECTHR_CONFIG,
    split=ECTHR_TRAIN_SPLIT,
    trust_remote_code=True,
)

try:
    eval_cases = load_dataset(
        ECTHR_DATASET,
        ECTHR_CONFIG,
        split=ECTHR_EVAL_SPLIT,
        trust_remote_code=True,
    )
except Exception:
    eval_cases = None

print(train_cases)
print('First labels:', train_cases[0]['labels'])
print('First normalized labels:', example_gold_articles(train_cases[0]))

In [ ]:
train_rows, train_stats = build_traversal_rows(train_cases, MAX_TRAIN_CASES, MAX_TRAIN_EXAMPLES, SEED)
eval_rows, eval_stats = build_traversal_rows(eval_cases, MAX_EVAL_CASES, MAX_EVAL_EXAMPLES, SEED + 1) if eval_cases is not None else ([], {})

raw_train_dataset = Dataset.from_list(train_rows)
raw_eval_dataset = Dataset.from_list(eval_rows) if eval_rows else None

print({'train': train_stats, 'eval': eval_stats})
print(raw_train_dataset[0]['messages'][1]['content'][:2200])
print(raw_train_dataset[0]['messages'][2]['content'])

## 7. Tokenize With Loss Only on the Assistant JSON

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'


def preprocess(example):
    messages = example['messages']
    prompt_messages = messages[:-1]

    prompt_text = tokenizer.apply_chat_template(
        prompt_messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    full_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )

    prompt_tokens = tokenizer(
        prompt_text,
        add_special_tokens=False,
        truncation=True,
        max_length=MAX_LENGTH,
    )
    full_tokens = tokenizer(
        full_text,
        add_special_tokens=False,
        truncation=True,
        max_length=MAX_LENGTH,
    )

    input_ids = full_tokens['input_ids']
    labels = input_ids.copy()
    prompt_len = min(len(prompt_tokens['input_ids']), len(labels))
    labels[:prompt_len] = [-100] * prompt_len

    return {
        'input_ids': input_ids,
        'attention_mask': full_tokens['attention_mask'],
        'labels': labels,
    }


train_dataset = raw_train_dataset.map(
    preprocess,
    remove_columns=raw_train_dataset.column_names,
    desc='Tokenizing traversal train examples',
)
train_dataset = train_dataset.filter(
    lambda example: any(label != -100 for label in example['labels']),
    desc='Dropping fully masked train examples',
)

eval_dataset = None
if raw_eval_dataset is not None:
    eval_dataset = raw_eval_dataset.map(
        preprocess,
        remove_columns=raw_eval_dataset.column_names,
        desc='Tokenizing traversal eval examples',
    )
    eval_dataset = eval_dataset.filter(
        lambda example: any(label != -100 for label in example['labels']),
        desc='Dropping fully masked eval examples',
    )

print({'train': len(train_dataset), 'eval': 0 if eval_dataset is None else len(eval_dataset)})

## 8. Load the Base Model in 4-bit and Attach LoRA

In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError('QLoRA training expects a CUDA GPU.')

torch.manual_seed(SEED)
compute_dtype = torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16

gc.collect()
torch.cuda.empty_cache()

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=compute_dtype,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quantization_config,
    torch_dtype=compute_dtype,
    low_cpu_mem_usage=True,
    device_map={'': 0},
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model)

peft_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias='none',
    target_modules='all-linear',
    task_type='CAUSAL_LM',
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

## 9. Train and Save the Adapter

In [ ]:
training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=NUM_TRAIN_EPOCHS,
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    logging_steps=10,
    save_strategy='epoch',
    save_total_limit=2,
    eval_strategy='epoch' if eval_dataset is not None else 'no',
    optim='paged_adamw_8bit',
    bf16=compute_dtype == torch.bfloat16,
    fp16=compute_dtype == torch.float16,
    warmup_ratio=0.03,
    lr_scheduler_type='constant',
    max_grad_norm=0.3,
    gradient_checkpointing=True,
    report_to='none',
    remove_unused_columns=False,
    seed=SEED,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=DataCollatorForSeq2Seq(
        tokenizer=tokenizer,
        model=model,
        padding=True,
        label_pad_token_id=-100,
    ),
)

trainer.train(resume_from_checkpoint=RESUME_FROM_CHECKPOINT)
trainer.save_model(str(OUTPUT_DIR))
tokenizer.save_pretrained(str(OUTPUT_DIR))

print(f'Saved QLoRA traversal adapter to: {OUTPUT_DIR}')

## 10. Smoke Test on One Traversal Decision

In [ ]:
def generate_reply(model, tokenizer, messages, max_new_tokens=384):
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors='pt',
        return_dict=True,
    )
    prompt = {key: value.to(model.device) for key, value in prompt.items()}

    model.eval()
    with torch.no_grad():
        generated_ids = model.generate(
            **prompt,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
        )

    input_len = prompt['input_ids'].shape[-1]
    new_tokens = generated_ids[0][input_len:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


test_messages = raw_train_dataset[0]['messages'][:-1]
print(generate_reply(model, tokenizer, test_messages))
print('Gold:', raw_train_dataset[0]['messages'][-1]['content'])

## 11. Load With `LocalRetrievalModelRuntime`

In [ ]:
from local_retrieval_model import load_local_retrieval_model

del trainer
del model
gc.collect()
torch.cuda.empty_cache()

runtime = load_local_retrieval_model(
    model_id=MODEL_ID,
    adapter_path=OUTPUT_DIR,
    use_4bit=True,
)

sample_user_prompt = raw_train_dataset[0]['messages'][1]['content']
reply = runtime.ask(
    sample_user_prompt,
    system_prompt=raw_train_dataset[0]['messages'][0]['content'],
    max_new_tokens=384,
    temperature=0.0,
)
print(reply)

## 12. ECtHR Batched Evaluation

This section evaluates whether the trained traversal adapter improves LATTICE retrieval on `alleged-violation-prediction`. It uses the reusable `EcthrTraversalEvaluator` module and can optionally compare against the base Qwen model with no adapter.

In [ ]:
# Free the ad-hoc runtime from the previous smoke test before LocalModelAPI loads its own runtime.
try:
    runtime.unload()
    del runtime
except NameError:
    pass

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


In [ ]:
import pickle

SRC_DIR = NOTEBOOK_DIR.parent
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from hyperparams import HyperParams
from llm_apis import LocalModelAPI
from llm_rl_playground.ecthr_evaluation import (
    EcthrTraversalEvaluator,
    get_label_names,
    load_ecthr_dataset,
    summarize_ecthr_cases,
)
from prompts import get_traversal_prompt_response_constraint
from tree_objects import SemanticNode
from utils import compute_node_registry, setup_logger

EVAL_TREE_PATH = NOTEBOOK_DIR.parents[1] / 'trees' / 'EU' / 'eu_conventions_notebook' / 'eu_conventions_tree-bottom-up-llm.pkl'
if not EVAL_TREE_PATH.exists():
    EVAL_TREE_PATH = TREE_PATH

tree_obj = pickle.loads(EVAL_TREE_PATH.read_bytes()) if EVAL_TREE_PATH.suffix == '.pkl' else json.loads(EVAL_TREE_PATH.read_text(encoding='utf-8'))
eval_semantic_root_node = SemanticNode().load_dict(tree_obj) if isinstance(tree_obj, dict) else tree_obj
eval_node_registry = compute_node_registry(eval_semantic_root_node)

eval_hp = HyperParams.from_args('--subset fiqa --tree_version eu_conventions_notebook')
eval_hp.TREE_PATH = str(EVAL_TREE_PATH)
eval_hp.DATASET = 'EU'
eval_hp.LLM_API_BACKEND = 'localModel'
eval_hp.LLM = MODEL_ID
eval_hp.LLM_API_TIMEOUT = 120
eval_hp.LLM_API_MAX_RETRIES = 4
eval_hp.LLM_MAX_CONCURRENT_CALLS = 1
eval_hp.LLM_API_STAGGERING_DELAY = 0.05
eval_hp.REASONING_IN_TRAVERSAL_PROMPT = -1
eval_hp.SUBSET = 'fiqa'
eval_hp.MAX_BEAM_SIZE = 8
eval_hp.SEARCH_WITH_PATH_RELEVANCE = True
eval_hp.NUM_LEAF_CALIB = 0
eval_hp.RELEVANCE_CHAIN_FACTOR = 0.5
eval_hp.MAX_PROMPT_PROTO_SIZE = None
eval_hp.MAX_DOC_DESC_CHAR_LEN = None

eval_logger = setup_logger('qwen_tree_traversal_ecthr_eval', str(OUTPUT_DIR / 'ecthr_eval.log'))

eval_llm_api_kwargs = {
    'max_concurrent_calls': eval_hp.LLM_MAX_CONCURRENT_CALLS,
    'response_mime_type': 'application/json',
    'response_schema': get_traversal_prompt_response_constraint(bool(eval_hp.REASONING_IN_TRAVERSAL_PROMPT)),
    'staggering_delay': eval_hp.LLM_API_STAGGERING_DELAY,
    'print_summary_report': False,
    'max_new_tokens': 384,
}

print(f'Loaded eval tree from: {EVAL_TREE_PATH}')
print(f'Eval tree nodes: {len(eval_node_registry)}')


In [ ]:
EVAL_SPLIT = ECTHR_EVAL_SPLIT
EVAL_N_CASES = 2
EVAL_START = 0
EVAL_NUM_ITERS = 3
EVAL_TOP_K_LEAVES = 6
EVAL_PREDICTION_MIN_SCORE = 0.4
EVAL_MAX_PREDICTED_ARTICLES = None
EVAL_USE_LLM_SELECTOR = False  # Keep False to measure traversal quality directly.
EVALUATE_BASE_MODEL = False

ecthr_eval_dataset = load_ecthr_dataset(split=EVAL_SPLIT, config=ECTHR_CONFIG)
ecthr_eval_label_names = get_label_names(ecthr_eval_dataset)

print(ecthr_eval_dataset)
print('First eval labels:', ecthr_eval_dataset[0]['labels'])


In [ ]:
def make_local_ecthr_evaluator(adapter_path):
    api = LocalModelAPI(
        MODEL_ID,
        logger=eval_logger,
        timeout=eval_hp.LLM_API_TIMEOUT,
        max_retries=eval_hp.LLM_API_MAX_RETRIES,
        adapter_path=None if adapter_path is None else str(adapter_path),
        use_4bit=True,
        serialize_requests=True,
        log_api_calls=False,
    )
    evaluator = EcthrTraversalEvaluator(
        semantic_root_node=eval_semantic_root_node,
        node_registry=eval_node_registry,
        hp=eval_hp,
        logger=eval_logger,
        llm_api=api,
        llm_api_kwargs=eval_llm_api_kwargs,
    )
    return evaluator


def run_ecthr_eval(label, adapter_path):
    print(f'\n================ Running {label} ================')
    evaluator = make_local_ecthr_evaluator(adapter_path)
    df, results = evaluator.evaluate_ecthr_cases_batched(
        ecthr_eval_dataset,
        ecthr_eval_label_names,
        n_cases=EVAL_N_CASES,
        num_iters=EVAL_NUM_ITERS,
        top_k=EVAL_TOP_K_LEAVES,
        start=EVAL_START,
        prediction_min_score=EVAL_PREDICTION_MIN_SCORE,
        max_predicted_articles=EVAL_MAX_PREDICTED_ARTICLES,
        use_llm_selector=EVAL_USE_LLM_SELECTOR,
    )
    summary = summarize_ecthr_cases(df)
    summary.insert(0, 'run', label)

    # Release the local model before another comparison run.
    evaluator.llm_api.runtime.unload()
    del evaluator
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return df, results, summary


In [ ]:
trained_eval_df, trained_eval_results, trained_eval_summary = run_ecthr_eval(
    'trained_tree_traversal_adapter',
    OUTPUT_DIR,
)

eval_summaries = [trained_eval_summary]

if EVALUATE_BASE_MODEL:
    base_eval_df, base_eval_results, base_eval_summary = run_ecthr_eval(
        'base_model_no_adapter',
        None,
    )
    eval_summaries.append(base_eval_summary)

comparison_df = __import__('pandas').concat(eval_summaries, ignore_index=True)
comparison_df


This is a smoke-test configuration. For a real comparison, increase `MAX_TRAIN_CASES`, `MAX_TRAIN_EXAMPLES`, `NUM_TRAIN_EPOCHS`, `EVAL_N_CASES`, and optionally set `EVALUATE_BASE_MODEL=True`.